# 08 · Reference-band clock stability

Is the next circadian cycle inside the amplitude-variant band of the previous one? The reference window gives z; θ_L is swept at fixed z to generate amplitude variants; their 5–95 % range (plus the model's residual noise) is the band the same recording is expected to stay in one period later.

Design (fixed in `src/analysis/band_stability.py` before results are seen):
- Traces: MSB recordings excluding U-2 OS. One cosinor period per trace. Every window starting ≥ 12 h is a reference; the target is the window one period later (known offset removed by a sample shift).
- Pair states: `conform` (inside the θ band), `amplitude` (outside the band, but the waveform is kept after free scaling), `break` (shape or phase changed), `no_rhythm` (reconstruction not above noise).
- Primary score per trace: `stable_frac` = fraction of pairs that are `conform` or `amplitude`. Also `hold_h` (time to the first two consecutive unstable pairs).
- Comparators on the same pairs: `acf` (model-free correlation one period apart), `log_amp`, and `self_sim` (reference reconstruction vs target).
- Evaluation: published circadian phenotype at cell-line level (n = 15).

Runs on CPU. Outputs: `results/band_stability/<run>/`.

In [ ]:
import os, sys
from pathlib import Path
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.analysis import band_stability as bs
pd.set_option("display.width", 220, "display.max_columns", 40)
%matplotlib inline

RUN = "s1/s1_MULTI_FC12_Z10_s0"   # stage/run_id of a finished sweep run
WHICH = "all"                      # "all" (train + validation) or "val"
BAND = "theta"                     # "theta" (z fixed) or "theta_z" (z drawn from the encoder posterior as well)

## 1. Pairs, bands and states

In [ ]:
pp, ex, amp = bs.run(RUN, which=WHICH, band=BAND)
pt = bs.per_trace(pp, amp)
pl = bs.per_line(pt)
cmp = bs.compare(pl)
summary = bs.summarise(pp, pt, cmp)
out = bs.save(RUN, pp, pt, pl, cmp, summary, band=BAND)
print("saved:", out)
pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)}).round(3)

## 2. Calibration

`calib_self` is the fraction of the reference input inside its own band and should be about 0.9 or higher. `theta_bound_frac` shows how often the best-fitting θ_L sits at the grid limit (a narrow θ range makes the band narrow).

In [ ]:
print(pd.Series(summary["state_fractions"]).round(3).to_string())
pt.groupby("phenotype")[["calib_self", "snr", "theta_bound_frac", "amp_ratio", "n_pairs"]].median().round(2)

## 3. Score vs published phenotype (cell-line level)

The band scores (`stable_frac`, `hold_h`) have to be at least as good as `acf` and `log_amp` to be worth using.

In [ ]:
cmp.round(3)

In [ ]:
fig = bs.plot_scores(pl)
fig.savefig(out / "scores_by_phenotype.png", dpi=150, bbox_inches="tight")

## 4. State timeline (every pair)

In [ ]:
fig = bs.plot_timeline(pp)
fig.savefig(out / "timeline.png", dpi=150, bbox_inches="tight")

## 5. Examples (one trace per phenotype)

In [ ]:
fig = bs.plot_examples(pp, ex)
fig.savefig(out / "examples.png", dpi=150, bbox_inches="tight")

## 6. Per cell line

In [ ]:
cols = ["cell_line", "phenotype", "n_traces", "stable_frac", "hold_h", "frac_conform", "frac_amplitude", "frac_break",
        "frac_no_rhythm", "conformity_theta", "conformity_shape", "self_sim", "acf", "log_amp"]
pl[cols].sort_values(["phenotype", "stable_frac"]).round(2)